# 01 langgraph state typed

## ¿Qué hace este notebook?

Demuestra el uso de un **objeto de estado tipado** en LangGraph. Define `SupportState`
(un `TypedDict`) que describe todos los campos del flujo de un agente de soporte: el
mensaje del usuario, la intención clasificada, las entidades detectadas, el borrador y
la respuesta final, además de campos de trazabilidad (`steps`, `confidence`).

El grafo es **lineal** con tres nodos:

1. `classify_intent` — clasifica el mensaje en `billing`, `bug`, `how_to` o `unknown`
   mediante palabras clave y extrae entidades.
2. `draft_reply` — redacta una respuesta acorde a la intención detectada.
3. `finalize` — fija la respuesta final.

Al compilar también genera un diagrama Mermaid (`01_langgraph_state_typed.png`). En
modo script se ejecuta como un bucle interactivo (REPL) que pide mensajes por consola.

## Ejemplo de uso

**Datos de interacción que espera el agente.** Este grafo es lineal y no se pausa: con la
entrada inicial concluye en una sola llamada `invoke`.

- Entrada inicial: `user_message` (str, obligatorio) y `steps` (lista, inicialízala en `[]`).
- No requiere reanudación: el resultado incluye `intent`, `confidence`, `entities` y
  `final_reply`.

```python
app = build_graph()
state = {"user_message": "I need a refund for invoice 12345", "steps": []}
result = app.invoke(state)            # concluye sin más interacción

print(result["final_reply"])
print("intent:", result["intent"], "| confidence:", result["confidence"])
print("entities:", result["entities"])
```

In [1]:
from __future__ import annotations

from typing import TypedDict, Literal, Optional, List
from langgraph.graph import StateGraph, END

In [2]:


#  Define a Typed State Object
class SupportState(TypedDict, total=False):
    # Input
    user_message: str

    # Computed fields
    intent: Literal["billing", "bug", "how_to", "unknown"]
    entities: List[str]

    # Draft + final
    draft_reply: str
    final_reply: str

    # Debug / tracing
    steps: List[str]
    confidence: float

In [3]:


def add_step(state: SupportState, note: str) -> SupportState:
    steps = state.get("steps", [])
    steps.append(note)
    state["steps"] = steps
    return state

In [4]:


#  Nodes (each node reads/writes typed state)
def classify_intent(state: SupportState) -> SupportState:
    msg = state["user_message"].lower()

    if any(k in msg for k in ["invoice", "refund", "payment", "billing"]):
        intent: SupportState["intent"] = "billing"
        confidence = 0.85
    elif any(k in msg for k in ["error", "crash", "bug", "stack trace"]):
        intent = "bug"
        confidence = 0.90
    elif any(k in msg for k in ["how do i", "how to", "steps", "guide"]):
        intent = "how_to"
        confidence = 0.80
    else:
        intent = "unknown"
        confidence = 0.50

    state["intent"] = intent
    state["confidence"] = confidence
    state["entities"] = extract_entities(state["user_message"])
    return add_step(state, f"classify_intent -> {intent} (conf={confidence})")

In [5]:


def extract_entities(text: str) -> List[str]:

    # Pull capitalized words and obvious product keywords
    tokens = text.replace(",", " ").replace(".", " ").split()
    entities = [t for t in tokens if t[:1].isupper() and len(t) > 2]
    for kw in ["LangGraph", "LangChain", "TaskFlow", "API", "Docker"]:
        if kw.lower() in text.lower() and kw not in entities:
            entities.append(kw)
    return entities

In [6]:


def draft_reply(state: SupportState) -> SupportState:
    intent = state.get("intent", "unknown")
    entities = state.get("entities", [])

    if intent == "billing":
        reply = (
            "Got it — billing issue. Please share your order/invoice ID and the last 4 digits of the card "
            "(never the full number). I'll help confirm the charge/refund status."
        )
    elif intent == "bug":
        reply = (
            "Sorry about the bug. Please share the exact error message and steps to reproduce. "
            "If you can, include your environment (OS, Python version) and a minimal snippet."
        )
    elif intent == "how_to":
        reply = (
            "Sure — I can walk you through it. Tell me what you're trying to achieve and your current setup, "
            "and I'll give step-by-step instructions."
        )
    else:
        reply = (
            "Thanks — I'm not fully sure yet. Can you clarify what you're trying to do and what went wrong?"
        )

    if entities:
        reply += f"\n\n(Detected keywords: {', '.join(entities)})"

    state["draft_reply"] = reply
    return add_step(state, "draft_reply -> created")

In [7]:


def finalize(state: SupportState) -> SupportState:
    # In a real workflow you might apply tone, compliance checks, or formatting rules here.
    state["final_reply"] = state.get("draft_reply", "")
    return add_step(state, "finalize -> final_reply set")

In [8]:

#  Build the graph
def build_graph():
    g = StateGraph(SupportState)

    g.add_node("classify_intent", classify_intent)
    g.add_node("draft_reply", draft_reply)
    g.add_node("finalize", finalize)

    g.set_entry_point("classify_intent")
    g.add_edge("classify_intent", "draft_reply")
    g.add_edge("draft_reply", "finalize")
    g.add_edge("finalize", END)

    graph = g.compile()

    # draw_mermaid_png() returns PNG bytes; pass the destination via the
    # keyword-only output_file_path argument (it is not a positional arg).
    try:
        graph.get_graph().draw_mermaid_png(
            output_file_path="01_langgraph_state_typed.png"
        )
    except Exception as e:
        # Rendering uses a remote API and may be unavailable offline.
        print(f"(skipped graph image: {e})")

    return graph

In [9]:


def main(app):


    print("Typed State Object")
    print("Type something and press Enter. Type 'exit' to quit.\n")

    while True:
        user = input("You: ").strip()
        if not user:
            continue
        if user.lower() in {"exit", "quit"}:
            print("Bye!")
            break

        initial_state: SupportState = {
            "user_message": user,
            "steps": [],
        }

        result: SupportState = app.invoke(initial_state)

        print("\n--- FINAL REPLY ---")
        print(result.get("final_reply", ""))

        print("\n--- STATE (typed fields) ---")
        for k in ["intent", "confidence", "entities", "steps"]:
            print(f"{k}: {result.get(k)}")
        print()

In [10]:
app = build_graph()
main(app)

Typed State Object
Type something and press Enter. Type 'exit' to quit.


--- FINAL REPLY ---
Thanks — I'm not fully sure yet. Can you clarify what you're trying to do and what went wrong?

(Detected keywords: Hola)

--- STATE (typed fields) ---
intent: unknown
confidence: 0.5
entities: ['Hola']
steps: ['classify_intent -> unknown (conf=0.5)', 'draft_reply -> created', 'finalize -> final_reply set']


--- FINAL REPLY ---
Thanks — I'm not fully sure yet. Can you clarify what you're trying to do and what went wrong?

--- STATE (typed fields) ---
intent: unknown
confidence: 0.5
entities: []
steps: ['classify_intent -> unknown (conf=0.5)', 'draft_reply -> created', 'finalize -> final_reply set']

Bye!
